# HyperMem Qwen14B Debug Run

Colab-safe debug run for the original HyperMem baseline using local vLLM Qwen2.5 14B chat and a separate vLLM Qwen3 embedding service. Reranker is disabled by default. Full baseline is gated off.

## Mount Drive

In [1]:
from google.colab import drive
drive.mount("/content/drive")


Mounted at /content/drive


## Go To HyperMem Repo

In [2]:
%cd "/content/drive/MyDrive/EverOS-main/methods/HyperMem"


/content/drive/.shortcut-targets-by-id/1vBRLfErjfFuozODKCRG1x0Zjj6xPBO37/EverOS-main/methods/HyperMem


## Install Dependencies

In [3]:
import subprocess
import sys

REPO_ROOT = "/content/drive/MyDrive/EverOS-main/methods/HyperMem"

subprocess.run([sys.executable, "-m", "pip", "install", "-U", "pip"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", f"{REPO_ROOT}/requirements.txt"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-U", "vllm", "openai"], check=True)


CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '-U', 'vllm', 'openai'], returncode=0)

## Prepare Dataset

In [4]:
import subprocess
import sys

REPO_ROOT = "/content/drive/MyDrive/EverOS-main/methods/HyperMem"
subprocess.run(
    [sys.executable, f"{REPO_ROOT}/scripts/prepare_dataset.py", "--src", "/content/drive/MyDrive/locomo10.json"],
    cwd=REPO_ROOT,
    check=True,
)


CompletedProcess(args=['/usr/bin/python3', '/content/drive/MyDrive/EverOS-main/methods/HyperMem/scripts/prepare_dataset.py', '--src', '/content/drive/MyDrive/locomo10.json'], returncode=0)

## Set Environment Variables

In [5]:
import os

REPO_ROOT = "/content/drive/MyDrive/EverOS-main/methods/HyperMem"
RUNTIME_DIR = "/content/hypermem_runtime"
os.makedirs(RUNTIME_DIR, exist_ok=True)

os.environ["HYPERMEM_DATA_PATH"] = "/content/drive/MyDrive/locomo10.json"
os.environ["HYPERMEM_LLM_PROVIDER"] = "vllm"
os.environ["VLLM_BASE_URL"] = "http://localhost:8000/v1"
os.environ["VLLM_API_KEY"] = "EMPTY"
os.environ["VLLM_MODEL_NAME"] = "Qwen/Qwen2.5-14B-Instruct"
os.environ["EMBEDDING_BASE_URL"] = "http://localhost:11810/v1"
os.environ["EMBEDDING_MODEL_NAME"] = "Qwen/Qwen3-Embedding-4B"
os.environ["HYPERMEM_USE_RERANKER"] = "false"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

os.chdir("/content")
print("Environment set. Working directory moved to /content to avoid Drive transport cwd issues.")


Environment set. Working directory moved to /content to avoid Drive transport cwd issues.


## Server Helpers

In [6]:
import json
import os
import socket
import subprocess
import time
import urllib.request
from pathlib import Path

def port_open(port: int) -> bool:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.settimeout(1.0)
        return sock.connect_ex(("127.0.0.1", port)) == 0

def models_ready(base_url: str) -> bool:
    try:
        with urllib.request.urlopen(base_url.rstrip("/") + "/models", timeout=5) as response:
            payload = json.loads(response.read().decode("utf-8"))
        return bool(payload.get("data"))
    except Exception:
        return False

def print_log_tail(log_path: str, n: int = 60) -> None:
    path = Path(log_path)
    if path.exists():
        lines = path.read_text(encoding="utf-8", errors="ignore").splitlines()[-n:]
        print("\n".join(lines))
    else:
        print(f"Log not found: {log_path}")

def wait_for_service(name: str, base_url: str, log_path: str, process: subprocess.Popen, timeout_s: int) -> None:
    deadline = time.time() + timeout_s
    while time.time() < deadline:
        if models_ready(base_url):
            print(f"{name} ready: {base_url}")
            return
        if process.poll() is not None:
            print(f"{name} exited early with code {process.returncode}")
            break
        time.sleep(10)
    print(f"{name} startup failed. Last log lines from {log_path}:")
    print_log_tail(log_path)
    raise RuntimeError(f"{name} not ready")


## Start vLLM Chat Server

In [ ]:
chat_base_url = os.environ["VLLM_BASE_URL"]
chat_log = "/content/hypermem_runtime/vllm_chat.log"

if models_ready(chat_base_url):
    print("Chat server already running.")
elif port_open(8000):
    raise RuntimeError("Port 8000 is in use, but /v1/models is not ready. Check existing process before starting chat server.")
else:
    chat_cmd = [
        "vllm", "serve", "Qwen/Qwen2.5-14B-Instruct",
        "--served-model-name", "Qwen/Qwen2.5-14B-Instruct",
        "--host", "0.0.0.0",
        "--port", "8000",
        "--dtype", "auto",
        "--gpu-memory-utilization", "0.65",
        "--max-model-len", "16384",
    ]
    chat_log_handle = open(chat_log, "w", encoding="utf-8")
    chat_proc = subprocess.Popen(chat_cmd, stdout=chat_log_handle, stderr=subprocess.STDOUT, cwd="/content")
    wait_for_service("Chat server", chat_base_url, chat_log, chat_proc, timeout_s=2400)


## Start vLLM Embedding Server

In [ ]:
embedding_base_url = os.environ["EMBEDDING_BASE_URL"]
embedding_log = "/content/hypermem_runtime/vllm_embedding.log"

if models_ready(embedding_base_url):
    print("Embedding server already running.")
elif port_open(11810):
    raise RuntimeError("Port 11810 is in use, but /v1/models is not ready. Check existing process before starting embedding server.")
else:
    embedding_cmd = [
        "vllm", "serve", "Qwen/Qwen3-Embedding-4B",
        "--served-model-name", "Qwen/Qwen3-Embedding-4B",
        "--host", "0.0.0.0",
        "--port", "11810",
        "--task", "embed",
        "--dtype", "auto",
        "--gpu-memory-utilization", "0.25",
    ]
    embedding_log_handle = open(embedding_log, "w", encoding="utf-8")
    embedding_proc = subprocess.Popen(embedding_cmd, stdout=embedding_log_handle, stderr=subprocess.STDOUT, cwd="/content")
    wait_for_service("Embedding server", embedding_base_url, embedding_log, embedding_proc, timeout_s=1800)


## Smoke Tests

In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, "scripts/test_vllm_provider.py"], cwd=REPO_ROOT, check=True)
subprocess.run([sys.executable, "scripts/test_embedding_provider.py"], cwd=REPO_ROOT, check=True)


## Compile Check

In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, "-m", "compileall", "hypermem"], cwd=REPO_ROOT, check=True)


## Run Debug Baseline

In [ ]:
import subprocess

subprocess.run(["bash", "scripts/colab_run_debug_baseline.sh"], cwd=REPO_ROOT, check=True)


## Verify Artifacts

In [ ]:
import json
from pathlib import Path

results_root = Path(REPO_ROOT) / "results"
matches = sorted(results_root.glob("*hypermem_qwen14b_debug*"))
if not matches:
    matches = sorted(results_root.glob("**/*hypermem_qwen14b_debug*"))

wanted = ["search_results.json", "responses.json", "judged.json"]
found = {}
for base in matches:
    if not base.is_dir():
        continue
    for name in wanted:
        path = base / name
        if path.exists():
            found[name] = path

for name in wanted:
    path = found.get(name)
    if path is None:
        print(f"Missing: {name}")
        continue
    print(f"{name}: {path}")
    print(f"  size: {path.stat().st_size} bytes")
    try:
        data = json.loads(path.read_text(encoding="utf-8"))
        if isinstance(data, dict):
            print(f"  keys: {list(data.keys())[:10]}")
        elif isinstance(data, list):
            print(f"  list length: {len(data)}")
    except Exception as e:
        print(f"  JSON read failed: {e}")


## Judged Summary

In [ ]:
import json
from pathlib import Path

judged_path = None
for path in (Path(REPO_ROOT) / "results").glob("**/judged.json"):
    if "hypermem_qwen14b_debug" in str(path):
        judged_path = path
        break

if judged_path is None:
    print("judged.json not found")
else:
    print(f"judged.json: {judged_path}")
    data = json.loads(judged_path.read_text(encoding="utf-8"))
    items = []
    for group_id, group_items in data.items():
        for item in group_items:
            items.append((group_id, item))
    print(f"judged examples: {len(items)}")
    categories = sorted({item.get("category") for _, item in items if item.get("category") is not None})
    print(f"categories: {categories}")
    scores = []
    for _, item in items:
        judgments = item.get("llm_judgments", {})
        if judgments:
            values = [1 if bool(v) else 0 for v in judgments.values()]
            scores.append(sum(values) / len(values))
    print(f"simple average score: {sum(scores) / len(scores):.4f}" if scores else "simple average score: n/a")
    if items:
        sample = items[0][1]
        print("sample question:", sample.get("question"))
        print("sample answer:", sample.get("answer"))
        print("sample golden_answer:", sample.get("golden_answer"))
        print("sample judgments:", sample.get("llm_judgments"))


## Full Baseline, Gated

In [ ]:
import subprocess

RUN_FULL_BASELINE = False
if RUN_FULL_BASELINE:
    subprocess.run(["bash", "scripts/colab_run_full_baseline.sh"], cwd=REPO_ROOT, check=True)
else:
    print("Full baseline is disabled. Set RUN_FULL_BASELINE=True only after debug baseline passes.")


## Troubleshooting

- If OOM: reduce chat gpu memory to 0.55, embedding to 0.20, max-model-len to 8192.
- If embedding fails: check `EMBEDDING_BASE_URL` and `/content/hypermem_runtime/vllm_embedding.log`.
- If chat fails: check `/content/hypermem_runtime/vllm_chat.log`.
- If dataset missing: check `/content/drive/MyDrive/locomo10.json`.
- If reranker issue: keep `HYPERMEM_USE_RERANKER=false`.
- If Drive transport disconnects: rerun Mount Drive, then rerun Set Environment Variables before retrying.